In [18]:
import pandas as pd
import numpy as np

df = pd.read_csv("../Data/FixTimeBars/EURUSD_H1_Demo_READY.csv",
                 index_col="time", parse_dates=True)

print(df["real_volume"].describe())
print(df["tick_volume"].describe())

count    8323.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: real_volume, dtype: float64
count     8323.000000
mean      4159.322119
std       3918.845374
min          1.000000
25%       1690.000000
50%       2895.000000
75%       5165.500000
max      49611.000000
Name: tick_volume, dtype: float64


In [19]:
# Typical price: the price used for weighting
df["tp"] = (df["high"] + df["low"] + df["close"]) / 3

# Daily VWAP: resets each calendar day (UTC)
day = df.index.date
df["vwap_daily"] = (
    (df["tp"] * df["tick_volume"]).groupby(day).cumsum()
    / df["tick_volume"].groupby(day).cumsum()
)

# Rolling VWAP: last 20 bars, never resets
df["vwap_roll"] = (
    (df["tp"] * df["tick_volume"]).rolling(20).sum()
    / df["tick_volume"].rolling(20).sum()
)

df[["close", "vwap_daily", "vwap_roll"]].head(30)

,close,vwap_daily,vwap_roll
time,,,
2025-05-16 04:00:00,1.12028,1.120290,NaN
2025-05-16 05:00:00,1.12016,1.120287,NaN
2025-05-16 06:00:00,1.11999,1.120237,NaN
2025-05-16 07:00:00,1.12026,1.120206,NaN
2025-05-16 08:00:00,1.12151,1.120397,NaN
2025-05-16 09:00:00,1.12118,1.120578,NaN
2025-05-16 10:00:00,1.12082,1.120664,NaN
2025-05-16 11:00:00,1.11986,1.120585,NaN
2025-05-16 12:00:00,1.11940,1.120492,NaN


In [20]:
dist = (df["close"] - df["vwap_daily"]).abs() / df["vwap_daily"]
print(dist.describe())
print()
print(dist.quantile([0.1, 0.25, 0.5, 0.75, 0.9]))

count    8323.000000
mean        0.000937
std         0.001048
min         0.000000
25%         0.000236
50%         0.000603
75%         0.001253
max         0.009986
dtype: float64

0.10    0.000091
0.25    0.000236
0.50    0.000603
0.75    0.001253
0.90    0.002180
dtype: float64


In [21]:
df["hour"] = df.index.hour
df["dist_daily"] = (df["close"] - df["vwap_daily"]).abs() / df["vwap_daily"]

print(df.groupby("hour")["dist_daily"].median().round(6))

hour
0     0.000107
1     0.000159
2     0.000228
3     0.000317
4     0.000348
5     0.000399
6     0.000409
7     0.000446
8     0.000418
9     0.000561
10    0.000619
11    0.000698
12    0.000781
13    0.000810
14    0.000878
15    0.001057
16    0.000951
17    0.001197
18    0.001171
19    0.001200
20    0.001159
21    0.001200
22    0.001308
23    0.001210
Name: dist_daily, dtype: float64


In [22]:
df["dist_roll"] = (df["close"] - df["vwap_roll"]).abs() / df["vwap_roll"]
print(df.groupby("hour")["dist_roll"].median().round(6))
print()
print(df["dist_roll"].quantile([0.1, 0.25, 0.5, 0.75, 0.9]))

hour
0     0.001124
1     0.001159
2     0.001110
3     0.001075
4     0.001020
5     0.001000
6     0.001009
7     0.000981
8     0.000892
9     0.000969
10    0.000906
11    0.000876
12    0.000933
13    0.000937
14    0.001012
15    0.001116
16    0.001006
17    0.001247
18    0.001181
19    0.001193
20    0.001153
21    0.001163
22    0.001251
23    0.001186
Name: dist_roll, dtype: float64

0.10    0.000196
0.25    0.000507
0.50    0.001071
0.75    0.001855
0.90    0.002966
Name: dist_roll, dtype: float64


In [23]:
def vwap_episodes(df, vwap_col="vwap_roll", band=0.0010, n_limit=10):
    vwap  = df[vwap_col].values
    close = df["close"].values
    idx   = df.index

    upper  = vwap * (1 + band)
    lower  = vwap * (1 - band)
    inside = (close < upper) & (close > lower)
    side   = np.sign(close - vwap)      # +1 above VWAP, -1 below

    episodes = []
    i, n = 1, len(df)

    while i < n:
        if np.isnan(vwap[i]) or np.isnan(vwap[i-1]):
            i += 1
            continue

        # episode starts: was outside last bar, inside now
        if inside[i] and not inside[i-1]:
            entry_side = side[i-1]
            if entry_side == 0:
                i += 1
                continue

            outcome, exit_i = "stayed", None
            for j in range(i+1, min(i + n_limit + 1, n)):
                if not inside[j]:
                    outcome = "bounce" if side[j] == entry_side else "break"
                    exit_i = j
                    break
            if exit_i is None:
                exit_i = min(i + n_limit, n - 1)

            episodes.append({
                "entry_time": idx[i],
                "bars": exit_i - i,
                "entry_side": entry_side,
                "outcome": outcome,
            })
            i = exit_i + 1
        else:
            i += 1

    return pd.DataFrame(episodes)


In [24]:
results = []
for band in [0.0005, 0.0010, 0.0015, 0.0020]:
    for n_limit in [5, 10, 24]:
        ep = vwap_episodes(df, "vwap_roll", band, n_limit)
        pct = ep["outcome"].value_counts(normalize=True) * 100
        results.append({
            "band_%":      band * 100,
            "n_limit":     n_limit,
            "episodes":    len(ep),
            "bounce_%":    round(pct.get("bounce", 0), 1),
            "break_%":     round(pct.get("break", 0), 1),
            "stayed_%":    round(pct.get("stayed", 0), 1),
            "median_bars": ep["bars"].median(),
        })

print(pd.DataFrame(results).to_string(index=False))

 band_%  n_limit  episodes  bounce_%  break_%  stayed_%  median_bars
   0.05        5       818      49.4     41.9       8.7          2.0
   0.05       10       818      52.0     46.3       1.7          2.0
   0.05       24       818      52.7     47.2       0.1          2.0
   0.10        5       793      47.0     23.7      29.3          3.0
   0.10       10       793      54.7     33.0      12.2          3.0
   0.10       24       793      59.9     39.0       1.1          3.0
   0.15        5       620      42.4     13.4      44.2          4.0
   0.15       10       620      50.3     22.7      26.9          4.0
   0.15       24       620      60.0     32.3       7.7          4.0
   0.20        5       434      39.4      4.4      56.2          5.0
   0.20       10       434      45.2     12.4      42.4          8.0
   0.20       24       434      57.8     23.7      18.4          8.0


In [25]:
df["fake_level"] = df["close"].shift(50)
print(df.columns.tolist())

['open', 'high', 'low', 'close', 'tick_volume', 'spread', 'real_volume', 'low_time', 'high_time', 'tp', 'vwap_daily', 'vwap_roll', 'hour', 'dist_daily', 'dist_roll', 'fake_level']


In [38]:
def vwap_episodes_ft(df, level_col="vwap_roll", band=0.0015,
                     n_limit=24, ft_bars=10):
    lvl   = df[level_col].values
    close = df["close"].values
    idx   = df.index

    upper  = lvl * (1 + band)
    lower  = lvl * (1 - band)
    inside = (close < upper) & (close > lower)
    side   = np.sign(close - lvl)

    episodes = []
    i, n = 1, len(df)

    while i < n:
        if np.isnan(lvl[i]) or np.isnan(lvl[i-1]):
            i += 1
            continue

        if inside[i] and not inside[i-1]:
            entry_side = side[i-1]
            if entry_side == 0:
                i += 1
                continue

            outcome, exit_i = "stayed", None
            for j in range(i+1, min(i + n_limit + 1, n)):
                if not inside[j]:
                    outcome = "bounce" if side[j] == entry_side else "break"
                    exit_i = j
                    break
            if exit_i is None:
                exit_i = min(i + n_limit, n - 1)

            ft = np.nan
            if outcome != "stayed":
                k = min(exit_i + ft_bars, n - 1)
                raw = (close[k] - close[exit_i]) / close[exit_i]
                ft  = raw * side[exit_i] * 10000

            episodes.append({
                "entry_time": idx[i],
                "bars":       exit_i - i,
                "outcome":    outcome,
                "ft_pips":    ft,
            })
            i = max(exit_i + 1, i + 1)
        else:
            i += 1

    return pd.DataFrame(episodes)

In [39]:
for col in ["vwap_roll", "fake_level"]:
    ep = vwap_episodes_ft(df, col, band=0.0015, n_limit=24, ft_bars=10)
    print(f"\n=== {col} ===")
    print(ep.groupby("outcome")["ft_pips"]
            .agg(["count", "mean", "median", "std"]).round(2))


=== vwap_roll ===
         count  mean  median    std
outcome                            
bounce     372 -1.80   -2.66  26.43
break      200  0.06   -1.59  27.31
stayed       0   NaN     NaN    NaN

=== fake_level ===
         count  mean  median    std
outcome                            
bounce     349  3.28    0.95  23.36
break      132 -3.71   -2.71  23.76
stayed       0   NaN     NaN    NaN


In [ ]:
# nothing found to prove that vwap indicator is working in this case, but the code is working and can be used for other indicators
# now i will try to do it on a ema 50 and the vwap to test to see if using both and making a range can make any sort of signal.


In [40]:
import ta
df["ema_50"] = ta.trend.EMAIndicator(close=df["close"], window=50).ema_indicator()

df["zone_top"] = df[["vwap_roll", "ema_50"]].max(axis=1)
df["zone_bot"] = df[["vwap_roll", "ema_50"]].min(axis=1)
df["gap_pips"] = (df["zone_top"] - df["zone_bot"]) / df["close"] * 10000

print(df["gap_pips"].describe().round(2))

count    8304.00
mean       11.24
std         9.46
min         0.00
25%         4.25
50%         8.91
75%        15.58
max        64.37
Name: gap_pips, dtype: float64


In [43]:
def zone_episodes(df, top_col="zone_top", bot_col="zone_bot",
                  n_limit=24, ft_bars=10):
    top   = df[top_col].values
    bot   = df[bot_col].values
    close = df["close"].values
    gap   = df["gap_pips"].values
    idx   = df.index

    inside = (close < top) & (close > bot)
    side   = np.where(close >= top, 1, np.where(close <= bot, -1, 0))

    episodes = []
    i, n = 1, len(df)

    while i < n:
        if np.isnan(top[i]) or np.isnan(top[i-1]):
            i += 1
            continue

        if inside[i] and not inside[i-1]:
            entry_side = side[i-1]
            if entry_side == 0:
                i += 1
                continue

            outcome, exit_i = "stayed", None
            for j in range(i+1, min(i + n_limit + 1, n)):
                if not inside[j]:
                    outcome = "bounce" if side[j] == entry_side else "break"
                    exit_i = j
                    break
            if exit_i is None:
                exit_i = min(i + n_limit, n - 1)

            ft = np.nan
            if outcome != "stayed":
                k = min(exit_i + ft_bars, n - 1)
                raw = (close[k] - close[exit_i]) / close[exit_i]
                ft  = raw * side[exit_i] * 10000

            episodes.append({
                "entry_time": idx[i],
                "gap_pips":   gap[i],
                "bars":       exit_i - i,
                "outcome":    outcome,
                "ft_pips":    ft,
            })
            i = max(exit_i + 1, i + 1)
        else:
            i += 1

    return pd.DataFrame(episodes)

In [44]:
ep = zone_episodes(df)
ep["gap_bucket"] = pd.qcut(ep["gap_pips"], 4)

print(len(ep), "episodes\n")
print(pd.crosstab(ep["gap_bucket"], ep["outcome"], normalize="index").round(3) * 100)
print()
print(ep.groupby(["gap_bucket", "outcome"])["ft_pips"].agg(["count","mean","std"]).round(2))

579 episodes

outcome           bounce  break  stayed
gap_bucket                             
(0.188, 6.223]      53.8   46.2     0.0
(6.223, 9.847]      52.4   47.6     0.0
(9.847, 16.556]     63.2   36.8     0.0
(16.556, 60.782]    65.5   33.8     0.7

                          count  mean    std
gap_bucket       outcome                    
(0.188, 6.223]   bounce      78  0.24  26.27
                 break       67 -1.61  21.88
                 stayed       0   NaN    NaN
(6.223, 9.847]   bounce      76  1.96  21.90
                 break       69 -1.13  26.97
                 stayed       0   NaN    NaN
(9.847, 16.556]  bounce      91 -1.96  26.60
                 break       53 -1.06  23.36
                 stayed       0   NaN    NaN
(16.556, 60.782] bounce      95 -0.92  34.69
                 break       49 -1.40  26.53
                 stayed       0   NaN    NaN


In [45]:
df["vwap_dist"] = (df["close"] - df["vwap_roll"]) / df["close"] * 10000  # pips
df["fwd_10"] = (df["close"].shift(-10) - df["close"]) / df["close"] * 10000

d = df.dropna(subset=["vwap_dist", "fwd_10"])
d["dist_bucket"] = pd.qcut(d["vwap_dist"], 10)

print(d.groupby("dist_bucket")["fwd_10"].agg(["count", "mean", "std"]).round(2))

                               count  mean    std
dist_bucket                                      
(-82.43700000000001, -20.355]    830  3.88  25.63
(-20.355, -12.943]               829  1.85  23.61
(-12.943, -8.625]                829  1.51  23.91
(-8.625, -4.698]                 830 -1.02  24.27
(-4.698, -0.801]                 829 -1.06  25.17
(-0.801, 3.234]                  829 -0.20  24.06
(3.234, 7.844]                   830  0.39  24.45
(7.844, 13.559]                  829 -1.76  26.15
(13.559, 22.443]                 829 -2.24  25.13
(22.443, 103.641]                830  2.13  23.79


C:\Users\steve\AppData\Local\Temp\ipykernel_2740\2488143202.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  d["dist_bucket"] = pd.qcut(d["vwap_dist"], 10)


In [46]:
df["fake_dist"] = (df["close"] - df["fake_level"]) / df["close"] * 10000

d = df.dropna(subset=["vwap_dist", "fake_dist", "fwd_10"]).copy()

for col in ["vwap_dist", "fake_dist"]:
    d["bucket"] = pd.qcut(d[col], 10)
    print(f"\n=== {col} ===")
    print(d.groupby("bucket", observed=True)["fwd_10"]
            .agg(["count", "mean", "std"]).round(2))


=== vwap_dist ===
                               count  mean    std
bucket                                           
(-82.43700000000001, -20.366]    827  3.86  25.63
(-20.366, -12.947]               826  1.78  23.52
(-12.947, -8.659]                826  1.42  23.97
(-8.659, -4.72]                  826 -1.08  24.03
(-4.72, -0.842]                  827 -1.26  25.23
(-0.842, 3.183]                  826 -0.20  23.90
(3.183, 7.826]                   826  0.04  24.17
(7.826, 13.54]                   826 -1.99  25.88
(13.54, 22.373]                  826 -2.25  25.25
(22.373, 103.641]                827  2.41  23.74

=== fake_dist ===
                    count  mean    std
bucket                                
(-203.072, -62.6]     827 -0.94  30.90
(-62.6, -38.454]      826  0.01  24.41
(-38.454, -23.316]    826 -0.59  22.63
(-23.316, -12.199]    826 -0.38  22.59
(-12.199, -0.689]     827 -0.06  23.54
(-0.689, 11.655]      826  2.73  23.02
(11.655, 24.843]      826  1.80  21.65
(24.843, 43

In [ ]:
mid = d.index[len(d) // 2]
print("split at:", mid, "\n")

for label, part in [("FIRST HALF", d.loc[:mid]), ("SECOND HALF", d.loc[mid:])]:
    part = part.copy()
    part["bucket"] = pd.qcut(part["vwap_dist"], 5)
    print(f"=== {label} ===")
    print(part.groupby("bucket", observed=True)["fwd_10"]
              .agg(["count", "mean", "std"]).round(2))
    print()

split at: 2026-01-20 00:00:00 

=== FIRST HALF ===
                               count  mean    std
bucket                                           
(-82.43700000000001, -13.144]    827  4.08  25.40
(-13.144, -4.301]                826  0.88  25.78
(-4.301, 4.276]                  826 -1.04  26.00
(4.276, 15.578]                  826 -1.23  24.56
(15.578, 100.683]                827  1.80  24.36

=== SECOND HALF ===
                    count  mean    std
bucket                                
(-69.908, -12.799]    827  1.19  23.74
(-12.799, -5.114]     826 -0.38  22.71
(-5.114, 2.153]       826 -0.90  23.49
(2.153, 12.031]       826 -0.21  23.51
(12.031, 103.641]     827 -1.40  25.94



In [50]:
import ta

df["rsi_14"] = ta.momentum.RSIIndicator(close=df["close"], window=14).rsi()

d = df.dropna(subset=["vwap_dist", "fwd_10", "rsi_14"]).copy()

print(d[["vwap_dist", "rsi_14"]].corr(method="spearman").round(3))

           vwap_dist  rsi_14
vwap_dist      1.000   0.919
rsi_14         0.919   1.000
